In [0]:
from datetime import date, datetime, timedelta
from pyspark.sql import functions as F

year = datetime.now().year
month = datetime.now().month
day = datetime.now().day 
print(year,month,day)

In [0]:
from datetime import date, timedelta

SOURCE_BASE = "/Volumes/workspace/default/klsescreener/klsescreener/"
start_date  = date(2026, 3, 1)
end_date    = date.today()

def path_exists(path):
    try:
        dbutils.fs.ls(path)
        return True
    except Exception:
        return False  # folder does not exist — skip it

# Build exact partition paths
paths = []
d = start_date
while d <= end_date:
    path = f"{SOURCE_BASE}/year={d.year}/month={d.month}/day={d.day}"
    if path_exists(path):
        paths.append(path)
    d += timedelta(days=1)

# Spark only reads these exact folders — nothing else
sdf = spark.read.parquet(*paths)
sdf.display() 

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.default.klsescreener (
    Name          STRING,
    Code          STRING,
    Category      STRING,
    Price         DOUBLE,
    Change        DOUBLE,
    ChangePercent STRING,
    `52week`      STRING,       -- backticks required: column starts with a number
    Volume        LONG,
    EPS           DOUBLE,
    DPS           DOUBLE,
    NTA           DOUBLE,
    PE            DOUBLE,
    DY            DOUBLE,
    ROE           DOUBLE,
    PTBV          DOUBLE,
    MCap_M        DOUBLE,
    Indicators    STRING,
    OtherInfo     DOUBLE,
    created_at    TIMESTAMP
)
USING DELTA
COMMENT 'klsescreener table'

In [0]:
sdf.createOrReplaceTempView('_sqldf')
 
spark.sql("""INSERT INTO workspace.default.klsescreener (
    Name, Code, Category, Price, Change, ChangePercent, `52week`, Volume,
     EPS, DPS, NTA, PE, DY, ROE, PTBV, MCap_M, Indicators, OtherInfo, created_at) 
     SELECT 
     Name, Code, Category, Price, Change, ChangePercent, `52week`, Volume,
     EPS, DPS, NTA, PE, DY, ROE, PTBV, MCap_M, Indicators, OtherInfo, created_at
     FROM _sqldf a 
     WHERE NOT EXISTS (SELECT 1 FROM workspace.default.klsescreener b WHERE a.Code = b.Code and a.created_at = b.created_at)""")

In [0]:
%sql
SELECT * FROM workspace.default.klsescreener